# MoLoRAG Reproduction: Baseline 1 (Local Text RAG)

**Baseline 1** using a fully local setup (No API keys needed):
1. **Local Embeddings**: `sentence-transformers/all-MiniLM-L6-v2`
2. **Local LLM**: `Qwen2.5-1.5B-Instruct`
3. **Subset**: First 5 PDFs of `MMLong` for speed (~10 mins total).

In [1]:
# 1. Setup Environment
!git clone https://github.com/WxxShirley/MoLoRAG.git
%cd MoLoRAG
!pip install "requests==2.32.4" transformers accelerate faiss-cpu langchain langchain-huggingface langchain-community pypdf PyMuPDF sentence-transformers huggingface_hub

fatal: destination path 'MoLoRAG' already exists and is not an empty directory.
/content/MoLoRAG


In [2]:
# 2. Download Subset (MMLong folder)
import os
from huggingface_hub import hf_hub_download, snapshot_download

os.makedirs('dataset/MMLong', exist_ok=True)
# Download the samples JSON
hf_hub_download(repo_id='xxwu/MoLoRAG', repo_type='dataset', filename='samples_MMLong.json', local_dir='./dataset/')

# Download MMLong folder contents
# Using 'MMLong/*' instead of specific prefixes to avoid FileNotFoundError
snapshot_download(repo_id='xxwu/MoLoRAG', repo_type='dataset',
                  allow_patterns=['MMLong/*'],
                  local_dir='./dataset/')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


samples_MMLong.json: 0.00B [00:00, ?B/s]

Fetching ... files: 0it [00:00, ?it/s]

'/content/MoLoRAG/dataset'

In [3]:
# 3. Create Local Baseline Scripts
rag_code = r'''import sys
import os
sys.path.append('../')
from utils import prepare_files, get_cur_time
from langchain_community.document_loaders import PyPDFLoader, UnstructuredFileLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from tqdm import tqdm
import argparse

def index_single_pdf(filepath, doc_id, embeddings, default_parser=True):
    if default_parser:
        loader = PyPDFLoader(filepath)
    else:
        loader = UnstructuredFileLoader(filepath)
    pages = loader.load_and_split()

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=args.chunk_size,
                                                   chunk_overlap=args.overlap,
                                                   add_start_index=True)
    chunks = text_splitter.split_documents(pages)

    cur_savepath = f"{save_dir}/{doc_id}"
    faiss_index = FAISS.from_documents(chunks, embeddings)
    faiss_index.save_local(cur_savepath)

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--dataset", type=str, default="MMLong")
    parser.add_argument("--save_dir", type=str, default="../tmp/tmp_dbs")
    parser.add_argument("--chunk_size", type=int, default=1000)
    parser.add_argument("--overlap", type=int, default=100)
    parser.add_argument("--model_name", type=str, default="sentence-transformers/all-MiniLM-L6-v2")
    args = parser.parse_args()

    save_dir = f"{args.save_dir}/{args.dataset}"
    os.makedirs(save_dir, exist_ok=True)

    print(f"{get_cur_time()} - Loading local embeddings: {args.model_name}")
    embeddings = HuggingFaceEmbeddings(model_name=args.model_name)

    pdf_files = prepare_files(root_dir=f"../dataset/{args.dataset}", suffix=".pdf")
    # Limit to 5 for fast testing as requested
    pdf_files = pdf_files[:5]

    for cur_pdf in tqdm(pdf_files, desc=f"Indexing (Subset) in {args.dataset}"):
        doc_id = cur_pdf.replace('.pdf', '')
        if os.path.exists(f"{save_dir}/{doc_id}"):
            continue

        try:
            index_single_pdf(filepath=f"../dataset/{args.dataset}/{cur_pdf}", doc_id=doc_id, embeddings=embeddings)
        except Exception as e:
            print(f"[ERROR] processing {cur_pdf}: {e}")

    print(f"{get_cur_time()} - Finished Indexing Subset! ")
'''
main_code = r'''import os
from langchain_community.vectorstores import FAISS
import json
import argparse
from tqdm import tqdm
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.prompts import ChatPromptTemplate
import torch
from transformers import pipeline
import time

def retrieve_context(query, index_folder, embeddings, top_k=5, max_length=1600):
    if not os.path.exists(index_folder):
        return ""

    faiss_index = FAISS.load_local(index_folder, embeddings, allow_dangerous_deserialization=True)
    searched_docs = faiss_index.similarity_search(query, k=top_k)
    context = []
    for doc in searched_docs:
        content = doc.page_content[:max_length].replace('\n', ' ').strip()
        context.append(' '.join(content.split()))
    return " ".join(context)

def main_llm_QA(args):
    st_time = time.time()

    # Load Embeddings
    embeddings = HuggingFaceEmbeddings(model_name=args.embed_model)

    # Load Local LLM
    print(f"Loading local LLM: {args.llm_model}")
    pipe = pipeline("text-generation", model=args.llm_model, device_map="auto", torch_dtype=torch.bfloat16)

    input_file = f"../dataset/samples_{args.dataset}.json"
    with open(input_file, 'r') as file:
        all_samples = json.load(file)

    # Filter for subset (only samples matching the first 5 PDFs)
    subset_docs = [f.replace('.pdf', '') for f in os.listdir(f"../dataset/{args.dataset}") if f.endswith('.pdf')][:5]
    samples = [s for s in all_samples if s['doc_id'].replace('.pdf', '') in subset_docs]

    for sample in tqdm(samples, desc="QA on Subset"):
        index_folder = f"../tmp/tmp_dbs/{args.dataset}/{sample['doc_id'].replace('.pdf', '')}"
        context_txt = retrieve_context(query=sample["question"], index_folder=index_folder, embeddings=embeddings, top_k=args.retrieve_topk)

        prompt = f"Context: {context_txt}\n\nQuestion: {sample['question']}\n\nAnswer concisely based on context:"

        result = pipe(prompt, max_new_tokens=128, do_sample=False)[0]['generated_text']
        response = result.replace(prompt, "").strip()
        sample[args.response_key] = response

    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    with open(output_file, 'w') as file:
        json.dump(samples, file, indent=4)

    print(f"Cost time: {(time.time() - st_time)/60:.2f} Mins")

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--dataset", type=str, default="MMLong")
    parser.add_argument("--llm_model", type=str, default="Qwen/Qwen2.5-1.5B-Instruct")
    parser.add_argument("--embed_model", type=str, default="sentence-transformers/all-MiniLM-L6-v2")
    parser.add_argument("--retrieve_topk", type=int, default=5)
    parser.add_argument("--response_key", type=str, default="raw_response")
    args = parser.parse_args()

    output_file = f"../results/{args.dataset}/LLM/local_qwen1.5b_top{args.retrieve_topk}.json"
    main_llm_QA(args)
'''

with open('LLMBaseline/rag_local.py', 'w') as f:
    f.write(rag_code)

with open('LLMBaseline/main_local.py', 'w') as f:
    f.write(main_code)


In [4]:
# 4. Indexing (Subset)
%cd LLMBaseline
!python3 rag_local.py --dataset MMLong

/content/MoLoRAG/LLMBaseline
03-08 16:15:12 - Loading local embeddings: sentence-transformers/all-MiniLM-L6-v2
modules.json: 100% 349/349 [00:00<00:00, 2.12MB/s]
config_sentence_transformers.json: 100% 116/116 [00:00<00:00, 652kB/s]
README.md: 10.5kB [00:00, 30.4MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 344kB/s]
config.json: 100% 612/612 [00:00<00:00, 4.37MB/s]
model.safetensors: 100% 90.9M/90.9M [00:01<00:00, 60.8MB/s]
Loading weights: 100% 103/103 [00:00<00:00, 1016.09it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
tokenizer_config.json: 100% 350/350 [00:00<00:00, 1.69MB/s]
vocab.txt: 232kB [00:00, 70.5MB/s]
tokenizer.json: 466kB [00:00, 115MB/s]
special

In [5]:
# 5. QA Inference (Subset)
!python3 main_local.py --dataset MMLong

Loading weights: 100% 103/103 [00:00<00:00, 1264.72it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading local LLM: Qwen/Qwen2.5-1.5B-Instruct
config.json: 100% 660/660 [00:00<00:00, 3.88MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors: 100% 3.09G/3.09G [00:32<00:00, 95.3MB/s]
Loading weights: 100% 338/338 [00:04<00:00, 74.57it/s, Materializing param=model.norm.weight] 
generation_config.json: 100% 242/242 [00:00<00:00, 1.60MB/s]
tokenizer_config.json: 7.30kB [00:00, 29.2MB/s]
vocab.json: 2.78MB [00:00, 121MB/s]
merges.txt: 1.67MB [00:00, 122MB/s]
tokenizer.json: 7.03MB [00:00, 151MB/s]
QA on Subset:   0% 0/48 [00:00<?, ?it/s]The following generatio

In [6]:
# 6. Results Overview
import json
import os
output_path = '../results/MMLong/LLM/local_qwen1.5b_top5.json'
if os.path.exists(output_path):
    with open(output_path, 'r') as f:
        results = json.load(f)
        print(f'Total results: {len(results)}')
        print('--- Sample Output ---')
        print(json.dumps(results[:2], indent=2))
else:
    print('Output file not found. Check previous steps for errors.')

Total results: 48
--- Sample Output ---
[
  {
    "doc_id": "reportq32015-151009093138-lva1-app6891_95.pdf",
    "doc_type": "Research report / Introduction",
    "question": "Regarding the global iOS breakdown, what percentage of the pie chart was iOS 9?",
    "answer": "51%",
    "evidence_pages": "[7]",
    "evidence_sources": "['Chart']",
    "answer_format": "Float",
    "id": "y2hqlb",
    "raw_response": "30%\n\nTo answer this question, we need to look at the given information about the global iOS breakdown. The pie chart provided shows the distribution of different versions of iOS across various regions and countries. To find out how much of the pie chart is dedicated to iOS 9, we would typically refer back to the specific data points or percentages associated with each version.\n\nGiven that the question asks for a concise answer based on the context provided, it's important to note that without direct access to the original source material (the pie chart), we cannot provide a